# 3b — ICI Specificity of CIDER T-cell Score × TMB: Overall Survival

**Purpose**: Builds Panel B of the combined `supplement_survival_specificity.png`
(assembled by `assemble_survival_specificity_figure.R`, alongside Panel A from
`2b_survival_specificity.ipynb`). Supplementary to Figure 5
(`3_survival_with_tmb.ipynb`): Figure 5's panel C crosses CIDER T-cell
High/Low with TMB High/Low in the ICI-treated cohort, showing the two
markers add independent prognostic value (stratified Cox HRs for CIDER
within each TMB stratum). This panel asks the specificity question for that
same CIDER × TMB design: does the CIDER-within-TMB-stratum effect hold
outside ICI therapy, or is it specific to ICI-treated patients?

**Cohort definitions** are identical to `15_ici_specificity.ipynb`'s
`parent_dfs` and this notebook's companion `2b_survival_specificity.ipynb`
(ICI Therapy, Non-ICI Therapy, Chemotherapy, Targeted therapy, Multiple
therapy, Hormonal therapy).

**CIDER High/Low threshold**: reused from Figure 4/5's ICI-treated split, the
same convention as `2b_survival_specificity.ipynb` -- see that notebook's
intro for the full rationale. "ICI Therapy" reuses `cpi_ready.rds`'s `group`
column directly; every other panel is classified relative to that same fixed
cutoff. `cpi_ready.rds` carries two rows per ICI-treated patient (their
natural cancer-type cohort_broad, plus a duplicate with cohort_broad forced
to "ICI-Treated" -- see `4_base.ipynb`); the threshold lookup deduplicates by
sampleId before joining, to avoid double-counting those patients.

**TMB High/Low**: `tmb_high` (from `0_prepare/3_meta.ipynb`'s `tmbStatus`), a
fixed clinical threshold rather than a per-cohort median split -- unlike
CIDER, this is an established external definition, so no cutoff-reuse
decision is needed; it's used identically across every panel.

**Panels (6 total, 3x2 layout):** ICI Therapy, Non-ICI Therapy, Chemotherapy,
Targeted therapy, Hormonal therapy, Multiple therapy. Each panel reproduces
Figure 5 panel C's design directly (not the separate 4-category
`survival_tmb_plot()` used for Figure 5's main per-cancer-type panel):
colour = CIDER T-cell Score (High/Low), linetype = TMB (High/Low), with the
overall log-rank p-value and the CIDER-within-TMB-stratum HRs (TMB Low, TMB
High) annotated on each panel.

No FDR correction is applied (6 descriptive KM panels, not a formal
hypothesis-testing forest plot) -- read the HRs/p-values as supporting visual
evidence, not as a replacement for a formal interaction test.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/ready.rds`, `cpi_ready.rds`, `clinical_update.txt` |
| **Outputs** | `figs/survival_specificity_tmb_panel.rds` (Panel B intermediate -- combined into `figs/supplement_survival_specificity.png` by `assemble_survival_specificity_figure.R`) |


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))   # ICI_TREATMENT_NORM, ICI_MECHANISM_MAP
source(paste0(dirname(getwd()), "/helpers/themes.r"))   # go_theme, high_low_color, figure_annotation, fmt_pval


In [ ]:
library(survival)
library(survminer)
library(patchwork)
library(broom)


## 1 — Data preparation


In [ ]:
# Load clinical metadata. Identical construction to 15_ici_specificity.ipynb
# and 2b_survival_specificity.ipynb, so treatment-type/mechanism
# classification matches those notebooks exactly.
# 'treatment_str' is renamed from 'treatment' to avoid collision with the
# existing 'treatment' column in ready.rds after the join.
clinical_raw <- data.table::fread(paste0(READY_DIR, "clinical_update.txt")) |>
  dplyr::transmute(
    sampleId       = sampleId,
    treatment_type = type,
    treatment_str  = treatment,
    mechanism_raw  = mechanism
  ) |>
  dplyr::distinct(sampleId, .keep_all = TRUE)


In [ ]:
# Load full ready object (T-cell group); join clinical metadata.
# ready.rds duplicates every patient: once with the natural cohort_broad and
# once with cohort_broad = 'ICI-Treated' (see 4_base.ipynb bind_rows call).
# Deduplicate by sampleId immediately to avoid inflated Ns.
# Classify non-ICI treatment regimens into mechanism sub-groups based on
# the treatment string. Returns NA for groups too small to be informative
# (e.g. irinotecan alone, ICI-containing combinations, anti-EGFR).
classify_non_ici_mech <- function(trt) {
  if (is.na(trt) || trt == "") return(NA_character_)
  t <- tolower(trt)
  has <- function(...) any(vapply(c(...), function(d) grepl(d, t, fixed = TRUE), logical(1)))
  # Exclude ICI-containing regimens from non-ICI sections
  if (has("pembrolizumab","nivolumab","atezolizumab","ipilimumab",
           "durvalumab","avelumab","tremelimumab")) return(NA_character_)
  # Targeted / biologic add-ons
  if (has("palbociclib","ribociclib","abemaciclib"))              return("CDK4/6i")
  if (has("trastuzumab","pertuzumab","lapatinib","ado-trastuzumab")) return("Chemo + Anti-HER2")
  if (has("bevacizumab","ramucirumab"))                           return("Chemo + Anti-VEGF")
  # Cetuximab/panitumumab: too few responders to model — drop
  if (has("cetuximab","panitumumab"))                             return(NA_character_)
  # Pure chemotherapy sub-groups (ordered from most specific to most general)
  has_plat <- grepl("carboplatin|cisplatin|oxaliplatin", t)
  has_gem  <- grepl("gemcitabine", t)
  has_tax  <- grepl("paclitaxel|docetaxel|cabazitaxel|nab-paclitaxel", t)
  has_fp   <- grepl("capecitabine|fluorouracil|trifluridine|tipiracil", t)
  has_anth <- grepl("doxorubicin|epirubicin", t)
  has_iri  <- grepl("irinotecan", t)
  has_etop <- grepl("etoposide", t)
  if (has_gem  && has_plat && !has_tax && !has_fp)  return("Gemcitabine + Platinum")
  if (has_tax  && has_plat)                         return("Taxane + Platinum")
  if (has_fp   && grepl("oxaliplatin", t) && !has_tax) return("Fluoropyrimidine + Oxaliplatin")
  # Irinotecan alone or with FP: too few responders — drop
  if (has_iri)                                      return(NA_character_)
  if (has_tax  && !has_plat)                        return("Taxane")
  if (has_fp   && !has_plat && !has_anth)           return("Fluoropyrimidine")
  if (has_anth)                                     return("Anthracycline")
  if (has_etop && has_plat)                         return("Etoposide + Platinum")
  if (has_plat && !has_gem && !has_tax && !has_fp)  return("Platinum")
  return(NA_character_)
}

ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::left_join(clinical_raw, by = "sampleId") |>
  dplyr::mutate(
    cider_log2   = log2(cider_score_depth_adj + 1),
    response   = as.integer(response),
    dcb        = as.integer(dcb),
    os_months  = osDays / 30.44,
    pfs_months = pfsEventDays / 30.44,
    cohort_pan = as.character(cohort_pan)
  ) |>
  dplyr::mutate(
    treatment_norm = unname(ICI_TREATMENT_NORM[treatment_str]),
    treatment_norm = dplyr::coalesce(treatment_norm, treatment_str)
  ) |>
  dplyr::mutate(
    ici_mechanism   = unname(ICI_MECHANISM_MAP[treatment_norm]),
    ici_mechanism   = dplyr::coalesce(ici_mechanism, "Other"),
    # ICI patients: use ICI mechanism map.
    # Chemo/Multiple therapy: reclassify by treatment string.
    # Targeted/Hormonal therapy: keep original mechanism_raw label.
    mechanism_group = dplyr::case_when(
      cpi == TRUE ~ ici_mechanism,
      treatment_type %in% c("Chemotherapy", "Multiple therapy") ~
        unname(vapply(treatment_str, classify_non_ici_mech, character(1))),
      TRUE ~ mechanism_raw
    )
  ) |>
  dplyr::select(-treatment_norm, -ici_mechanism) |>
  # Restrict to patients with response (BOR), OS, and TMB status annotated.
  dplyr::filter(!is.na(response), !is.na(os_months), !is.na(osEvent), !is.na(tmb_high))

message("Total unique patients: ", dplyr::n_distinct(ready$sampleId))


## 2 — Cohort definitions

Same treatment-class groups as `15_ici_specificity.ipynb`'s `parent_dfs` / `2b_survival_specificity.ipynb`.


In [ ]:
# Top-level treatment-class groups -- identical definitions to
# 15_ici_specificity.ipynb's parent_dfs, so cohort membership is consistent
# across all three supplementary specificity figures.
parent_dfs <- list(
  "ICI Therapy"      = ready |> dplyr::filter(cpi == TRUE) |>
                        dplyr::mutate(cohort_pan = as.character(cohort_broad)),
  "Non-ICI Therapy"  = ready |> dplyr::filter(cpi == FALSE),
  "Chemotherapy"     = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Chemotherapy"),
  "Targeted therapy" = ready |> dplyr::filter(cpi == FALSE,
                          treatment_type == "Targeted therapy" | mechanism_group == "CDK4/6i"),
  "Multiple therapy" = ready |> dplyr::filter(cpi == FALSE,
                          mechanism_group %in% c("Chemo + Anti-HER2", "Chemo + Anti-VEGF")),
  "Hormonal therapy" = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Hormonal therapy")
)

for (nm in names(parent_dfs)) {
  message(nm, ": n = ", nrow(parent_dfs[[nm]]), ", events (OS) = ", sum(parent_dfs[[nm]]$osEvent, na.rm = TRUE))
}


## 3 — CIDER High/Low threshold (reused from Figure 4) + TMB groups


In [ ]:
# Figure 4's own ICI-treated CIDER High/Low split (0_prepare/4_base.ipynb:
# ntile(cider_score_depth_adj, 2), grouped by gp only). Reused here as a
# single fixed cutoff for every panel, exactly as in
# 2b_survival_specificity.ipynb.
# ready.rds (and cpi_ready.rds, derived from it) duplicates every
# ICI-treated patient: once under their natural cohort_broad (e.g. "SKCM")
# and once with cohort_broad forced to "ICI-Treated" (4_base.ipynb's
# bind_rows call, for Figure 3/4's pooled-vs-per-cancer-type panels). Both
# copies have cpi == TRUE, so cpi_ready.rds carries two rows per ICI-treated
# patient for gp == "T-cell". Deduplicate by sampleId here -- group and
# cider_score_depth_adj are identical between the two copies -- otherwise
# joining against this below would double-count every ICI Therapy patient.
cpi_ready_group <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::select(sampleId, group, cider_score_depth_adj)

cider_cutoff <- max(
  cpi_ready_group$cider_score_depth_adj[cpi_ready_group$group == "Low"],
  na.rm = TRUE
)
message("CIDER High/Low cutoff (Figure 4's ICI-treated median): ", round(cider_cutoff, 3))

assign_cider_group <- function(df) {
  df |>
    dplyr::mutate(
      cider_group = factor(
        ifelse(cider_score_depth_adj > cider_cutoff, "High", "Low"),
        levels = c("Low", "High")
      )
    )
}

# ICI Therapy: reuse Figure 4's own group column directly (exact match, not
# a recomputation) via a join on sampleId.
join_ici_cider_group <- function(df) {
  n_before <- nrow(df)
  df <- df |>
    dplyr::inner_join(
      cpi_ready_group |> dplyr::select(sampleId, cider_group = group),
      by = "sampleId"
    )
  if (nrow(df) != n_before) {
    message("  (n changed ", n_before, " -> ", nrow(df), " after joining cpi_ready.rds group -- check for duplicates)")
  }
  df
}

# TMB High/Low: a fixed external clinical threshold (tmb_high, from
# tmbStatus in 3_meta.ipynb), not a median split -- applied identically to
# every panel, with no cutoff-reuse decision needed.
assign_tmb_group <- function(df) {
  df |> dplyr::mutate(tmb_group = factor(ifelse(tmb_high == 1, "High", "Low"), levels = c("Low", "High")))
}

panel_dfs <- list(
  "ICI Therapy"      = parent_dfs[["ICI Therapy"]]      |> join_ici_cider_group() |> assign_tmb_group(),
  "Non-ICI Therapy"  = parent_dfs[["Non-ICI Therapy"]]  |> assign_cider_group()   |> assign_tmb_group(),
  "Chemotherapy"     = parent_dfs[["Chemotherapy"]]     |> assign_cider_group()   |> assign_tmb_group(),
  "Targeted therapy" = parent_dfs[["Targeted therapy"]] |> assign_cider_group()   |> assign_tmb_group(),
  "Hormonal therapy" = parent_dfs[["Hormonal therapy"]] |> assign_cider_group()   |> assign_tmb_group(),
  "Multiple therapy" = parent_dfs[["Multiple therapy"]] |> assign_cider_group()   |> assign_tmb_group()
)

for (nm in names(panel_dfs)) {
  message(nm, ":")
  print(table(panel_dfs[[nm]]$cider_group, panel_dfs[[nm]]$tmb_group, dnn = c("CIDER", "TMB")))
}


## 4 — KM panel function


In [ ]:
MIN_N      <- 20
MIN_EVENTS <- 5

# One panel per cohort: CIDER High/Low (colour) x TMB High/Low (linetype)
# overall-survival curves, reproducing Figure 5 panel C's design
# (3_survival_with_tmb.ipynb's "2b - ICI-Treated single panel") for each
# treatment-class cohort instead of just ICI-Treated. Annotates the overall
# log-rank p-value and the CIDER-within-TMB-stratum HRs. Returns NULL (and
# skips) if the panel doesn't clear MIN_N/MIN_EVENTS or a model fails to fit.
tmb_cider_panel <- function(df, panel_label) {
  df <- df |>
    tidyr::drop_na(cider_group, tmb_group, os_months, osEvent) |>
    dplyr::filter(os_months > 0) |>
    dplyr::mutate(combined_group = interaction(cider_group, tmb_group, sep = " / CIDER "))

  if (nrow(df) < MIN_N || sum(df$osEvent, na.rm = TRUE) < MIN_EVENTS ||
      dplyr::n_distinct(df$cider_group) < 2 || dplyr::n_distinct(df$tmb_group) < 2) {
    message("Skipping ", panel_label, " (n=", nrow(df), ", events=", sum(df$osEvent, na.rm = TRUE), ")")
    return(NULL)
  }

  fit <- tryCatch(survival::survfit(Surv(os_months, osEvent) ~ cider_group + tmb_group, data = df), error = function(e) NULL)
  cox_overall <- tryCatch(survival::coxph(Surv(os_months, osEvent) ~ cider_group * tmb_group, data = df), error = function(e) NULL)
  if (is.null(fit) || is.null(cox_overall)) {
    message("Skipping ", panel_label, " (model fit failed)")
    return(NULL)
  }
  p_label <- fmt_pval(summary(cox_overall)$logtest[3])

  # Stratified HRs: CIDER High vs Low within each TMB stratum.
  fmt_hr_tmb <- function(tmb_grp) {
    sub_d <- df |> dplyr::filter(tmb_group == tmb_grp)
    if (nrow(sub_d) < MIN_N || dplyr::n_distinct(sub_d$cider_group) < 2 ||
        sum(sub_d$osEvent, na.rm = TRUE) < MIN_EVENTS) return(NULL)
    cx <- tryCatch(survival::coxph(Surv(os_months, osEvent) ~ cider_group, data = sub_d), error = function(e) NULL)
    if (is.null(cx)) return(NULL)
    hr <- round(exp(coef(cx)), 2)
    pv <- gsub("p = ", "p=", fmt_pval(summary(cx)$logtest[3]))
    paste0("HR=", hr, " (", pv, ")")
  }
  hr_tmb_low  <- fmt_hr_tmb("Low")
  hr_tmb_high <- fmt_hr_tmb("High")
  hr_lines <- paste(
    c(if (!is.null(hr_tmb_low))  paste0("CIDER within TMB Low:  ",  hr_tmb_low),
      if (!is.null(hr_tmb_high)) paste0("CIDER within TMB High: ", hr_tmb_high)),
    collapse = "\n"
  )

  km_df <- broom::tidy(fit) |>
    dplyr::mutate(
      cider_group = factor(ifelse(grepl("cider_group=High", strata), "High", "Low"), levels = c("Low", "High")),
      tmb_group   = factor(ifelse(grepl("tmb_group=High",   strata), "High", "Low"), levels = c("Low", "High"))
    )

  # Single stacked label (N, overall p, stratified HRs) in the upper-right
  # corner of the panel, matching Figure 5's annotation placement
  # (3_survival_with_tmb.ipynb's ICI-Treated single panel). x/y = Inf with
  # hjust/vjust > 1 anchors the text just inside the top-right corner
  # regardless of each panel's own time axis range.
  label_lines <- paste(
    c(paste0("N = ", nrow(df)),
      paste0("Overall: ", p_label),
      hr_lines),
    collapse = "\n"
  )

  ggplot2::ggplot(
    km_df,
    ggplot2::aes(x = time, y = estimate, colour = cider_group, linetype = tmb_group, group = strata)
  ) +
    ggplot2::geom_step(linewidth = 0.8) +
    ggplot2::scale_colour_manual(name = "CIDER T-cell Score", values = high_low_color) +
    ggplot2::scale_linetype_manual(name = "TMB", values = c("High" = "dashed", "Low" = "solid")) +
    ggplot2::coord_cartesian(ylim = c(0, 1)) +
    ggplot2::scale_x_continuous(limits = c(0, NA)) +
    go_theme +
    ggplot2::theme(
      plot.title      = ggplot2::element_text(hjust = 0.5),
      legend.position = "bottom",
      legend.box      = "vertical"
    ) +
    ggplot2::annotate("text", x = Inf, y = Inf, hjust = 1.05, vjust = 1.3,
                      label = label_lines, size = 2.6, colour = "grey30") +
    ggplot2::labs(title = panel_label, x = "Time (months)", y = "Survival Probability")
}


## 5 — Assemble figure


In [ ]:
PANEL_ORDER <- c(
  "ICI Therapy", "Non-ICI Therapy",
  "Chemotherapy", "Targeted therapy",
  "Hormonal therapy", "Multiple therapy"
)

panels <- purrr::imap(panel_dfs[PANEL_ORDER], tmb_cider_panel)
panels <- purrr::compact(panels)  # drop any panel that failed MIN_N/MIN_EVENTS or model fit

if (length(panels) < length(PANEL_ORDER)) {
  warning(length(PANEL_ORDER) - length(panels), " panel(s) skipped -- see messages above.")
}

# Enlarge the legend (25% larger than go_theme's defaults -- legend.text 9pt
# -> 11.25pt, legend.title 10pt -> 12.5pt, key size 1.2 lines -> 1.5 lines)
# by overriding each panel's own theme BEFORE wrap_plots(..., guides =
# "collect") merges them -- guides = "collect" pulls its single combined
# legend from the panels' own legend settings, so this is the safe way to
# resize it. Deliberately NOT done via patchwork's `&` operator on the
# already-combined object: that pattern is flagged elsewhere in this
# pipeline (20_ici_specificity_main.ipynb's Section 7 notes) as fragile
# across ggplot2/patchwork versions when combined with guides = "collect".
panels <- lapply(panels, function(p) {
  p + ggplot2::theme(
    legend.text     = ggplot2::element_text(size = 11.25, colour = COL_INK),
    legend.title    = ggplot2::element_text(size = 12.5, face = "bold", colour = COL_INK),
    legend.key.size = ggplot2::unit(1.5, "lines")
  )
})

# Intermediate panel only -- no standalone .png here. Combined with
# 2b_survival_specificity.ipynb's panel into the final
# supplement_survival_specificity.png by assemble_survival_specificity_figure.R
# (Panel A = the CIDER-only grid, Panel B = this grid).
survival_specificity_tmb_panel <-
  patchwork::wrap_plots(panels, ncol = 3, guides = "collect") +
  figure_annotation(
    "CIDER T-cell Score x TMB and Overall Survival: Specificity to ICI Therapy",
    tag = FALSE, legend_pos = "bottom"
  )

set_plot_dims(12, 8); survival_specificity_tmb_panel

saveRDS(survival_specificity_tmb_panel, paste0(CIDER_PROJ_DIR, "figs/survival_specificity_tmb_panel.rds"))
message("Saved survival_specificity_tmb_panel.rds (Panel B intermediate)")
